# 🔗 IntelliCode-SL | Classifier SLM — Merge Adapter into Base Model

**What this does:**  
Loads the base model and LoRA adapter both at full fp16 precision, merges them into a single unified model, and saves the merged model to Google Drive.

**Why merge?**  
- No adapter loading overhead at inference time
- Single model file — simpler to load and deploy
- Full fp16 precision throughout — no quantization
- Slightly faster inference since LoRA math is baked in

**Input:**  `MyDrive/IntelliCode-SL/adapters/classifier_adapter/`  
**Output:** `MyDrive/IntelliCode-SL/merged_models/classifier_merged/`

> T4 GPU runtime required. Run cells top to bottom.

In [ ]:
# ── Cell 1: Install ────────────────────────────────────────────
!pip install -q unsloth transformers peft accelerate
print("✅ Dependencies installed")

In [ ]:
# ── Cell 2: Imports ────────────────────────────────────────────
import os, torch
from google.colab import drive
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

os.environ['UNSLOTH_USE_MODELSCOPE'] = '1'

print("✅ Imports done")
print("GPU :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NOT FOUND")
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB")

In [ ]:
# ── Cell 3: Config + Drive + Login ─────────────────────────────
drive.mount("/content/drive")

HF_TOKEN     = "hf_XXXXXXXXXXXXXXXXXXXXXXXXXX"  # ← paste your token
MODEL_NAME   = "Qwen/Qwen2.5-Coder-0.5B-Instruct"
ADAPTER_PATH = "/content/drive/MyDrive/IntelliCode-SL/adapters/classifier_adapter"
SAVE_PATH    = "/content/drive/MyDrive/IntelliCode-SL/merged_models/classifier_merged"

login(token=HF_TOKEN)
os.makedirs(SAVE_PATH, exist_ok=True)

print("✅ Drive mounted + HF login done")
print(f"   Base model   : {MODEL_NAME}")
print(f"   Adapter path : {ADAPTER_PATH}")
print(f"   Save path    : {SAVE_PATH}")

In [ ]:
# ── Cell 4: Load Base Model at fp16 ───────────────────────────
print("Loading base model at fp16 (no quantization)...")

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype = torch.float16,
    device_map  = "auto",
    token       = HF_TOKEN,
)
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token = HF_TOKEN
)

print("✅ Base model loaded")
print(f"   VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# ── Cell 5: Load Adapter at fp16 ──────────────────────────────
print("Loading LoRA adapter at fp16...")

model_with_adapter = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH,
    torch_dtype = torch.float16
)

print("✅ Adapter loaded")
print(f"   VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# ── Cell 6: Merge Adapter into Base Model ─────────────────────
# merge_and_unload() mathematically bakes the LoRA weights
# into the base model weights and returns a plain model
# with no adapter — same as the base model class, just
# with the fine-tuned weights incorporated.

print("Merging adapter into base model...")

merged_model = model_with_adapter.merge_and_unload()

print("✅ Merge complete")
print(f"   VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"   Model type: {type(merged_model).__name__}")

In [ ]:
# ── Cell 7: Save Merged Model to Drive ────────────────────────
print(f"Saving merged model to Drive...")
print(f"   Path: {SAVE_PATH}")
print(f"   (This may take a few minutes)")

merged_model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print(f"✅ Merged model saved")

# Show what was saved
import os
files = os.listdir(SAVE_PATH)
print(f"\n  Files saved ({len(files)} total):")
for f in sorted(files):
    size_mb = os.path.getsize(os.path.join(SAVE_PATH, f)) / 1e6
    print(f"   {f:<40} {size_mb:>8.1f} MB")

In [ ]:
# ── Cell 8: Verify — Load Merged Model and Run Inference ───────
# Load the saved merged model fresh from Drive to confirm it works
import gc

print("Verifying saved model by loading fresh from Drive...")

# Free existing model from VRAM first
del merged_model, model_with_adapter, base_model
gc.collect()
torch.cuda.empty_cache()

# Load fresh from saved path
verify_model = AutoModelForCausalLM.from_pretrained(
    SAVE_PATH,
    torch_dtype = torch.float16,
    device_map  = "auto",
)
verify_tokenizer = AutoTokenizer.from_pretrained(SAVE_PATH)
verify_model.eval()

print("✅ Merged model loaded fresh from Drive")
print(f"   VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# ── Cell 9: Quick Inference Test ──────────────────────────────
PROMPT_TEMPLATE = """### Instruction:
Classify the following user request into exactly one category:
debug, generate, modify, explain, document, unknown

### User Request:
{}

### Category:
"""

test_cases = [
    ("Fix the NullPointerException in my Java code",          "debug"),
    ("Write a function to calculate fibonacci numbers",        "generate"),
    ("Add error handling to this function",                    "modify"),
    ("What does this recursive function do?",                  "explain"),
    ("Generate docstrings for my Python class",               "document"),
    ("What is the difference between TCP and UDP",             "unknown"),
    ("My loop runs forever, fix the infinite loop bug",        "debug"),
    ("Create a binary search tree class",                      "generate"),
    ("Refactor this code to use list comprehensions",          "modify"),
    ("Explain how this decorator works step by step",          "explain"),
]

VALID_LABELS = {"debug", "generate", "modify", "explain", "document", "unknown"}

print("Running inference test on merged model...\n")
correct = 0
for prompt, expected in test_cases:
    inputs = verify_tokenizer(
        PROMPT_TEMPLATE.format(prompt),
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to("cuda")
    with torch.no_grad():
        outputs = verify_model.generate(
            **inputs,
            max_new_tokens=10,
            do_sample=False,
            pad_token_id=verify_tokenizer.eos_token_id
        )
    decoded = verify_tokenizer.decode(outputs[0], skip_special_tokens=True)
    raw = decoded.split("### Category:")[-1].strip().split()[0].lower()
    predicted = next((v for v in VALID_LABELS if raw.startswith(v)), raw)
    match = predicted == expected
    if match:
        correct += 1
    print(f"  {'✅' if match else '❌'} Expected: {expected:<10} Got: {predicted:<10} | {prompt[:50]}...")

print(f"\n  Accuracy on spot check: {correct}/{len(test_cases)} ({correct/len(test_cases)*100:.0f}%)")
print(f"\n✅ Merged model verified and working!")
print(f"   Saved at: {SAVE_PATH}")